# SÓSIA no Colab

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ridd1kulusC0d3r/SOSIA/blob/claude/laughing-davinci-bj48ot/notebooks/sosia_colab.ipynb)

Proteção de marca/pessoa contra perfis que se passam por ela — **OSINT passivo, explicável e mensurável**.

> **Uso responsável:** apenas dados públicos, sem login, sem contornar CAPTCHA/bloqueios, rate limit por host.
> Use para marca/pessoa que **você representa**. Prioridade ≠ veredito jurídico; **revise manualmente antes de denunciar**.
> As células marcadas **🌐 rede** fazem requisições reais; as demais rodam offline.

## 1. Instalação

In [ ]:
BRANCH = "claude/laughing-davinci-bj48ot"  #@param {type:"string"}
!pip -q install "git+https://github.com/Ridd1kulusC0d3r/SOSIA@{BRANCH}"
import sosia; print("SÓSIA", sosia.__version__)

## 2. Configure a marca

In [ ]:
#@title Marca / pessoa protegida
BRAND   = "Marca Zeta"            #@param {type:"string"}
HANDLE  = "marcazeta"             #@param {type:"string"}
DOMAINS = "marcazeta.com.br"      #@param {type:"string"}
OFFICIAL = "marcazeta, marcazeta_sac"  #@param {type:"string"}
EXTRA_AFFIXES = "clube, cartao"   #@param {type:"string"}

from sosia.model import Brand
_split = lambda s: [x.strip() for x in s.split(",") if x.strip()]
brand = Brand(name=BRAND, handle=HANDLE, domains=_split(DOMAINS),
              official={"*": {h.lower() for h in _split(OFFICIAL)}}, extra_affixes=_split(EXTRA_AFFIXES))
brand

## 3. Variantes de username (offline)
Ranqueadas por *massa de probabilidade por família* — veja `docs/DESIGN.md`. Os pesos são hipóteses; re-estime com `sosia fit-priors`.

In [ ]:
from sosia.variants import VariantEngine
variants = VariantEngine(brand.handle, extra_affixes=brand.extra_affixes).generate(exclude=brand.official["*"])
print(f"{len(variants)} variantes")
rows = [(v.value, ", ".join(sorted(v.families)), v.similarity) for v in variants[:40]]
try:
    import pandas as pd; display(pd.DataFrame(rows, columns=["variante", "famílias", "semelhança"]))
except ImportError:
    for r in rows: print(r)

## 4. Ler uma bio (offline)
Sinais agrupados, com negação e mitigadores. **CRÍTICO** exige evidência independente — uma palavra solta nunca basta.

In [ ]:
from sosia.profiles import Profile, find_links
from sosia.signals import analyze_bio
from sosia.scoring import assess
from sosia.textutil import similarity

def triage(username, display_name, bio):
    p = Profile(display_name=display_name, bio=bio or None, links=find_links(bio or ""))
    rep = analyze_bio(p, brand)
    a = assess(brand, "*", username, similarity(brand.handle, username), 0.95, p, rep)
    print(f"{username}: {a.priority.value} · abuso={a.abuse_level} · severidade={a.severity} · vínculo={a.affiliation}")
    for r in a.reasons: print("   -", r)

triage("marcazeta_suporte", "Marca Zeta Suporte", "Suporte oficial Marca Zeta. Chame no WhatsApp e faça o Pix da taxa de liberação.")
triage("marcazeta_loja", "Zeta Revenda", "Revenda autorizada. Aceitamos Pix e cartão. Atendimento via WhatsApp.")
triage("marcazeta_sac2", "Marca Zeta", "Atendimento oficial. Nunca pedimos senha ou Pix por aqui.")

## 5. 🌐 Calibrar e validar os conectores no *seu* ambiente
O resultado depende de IP/região. Só plataformas `trusted`/validadas entram na varredura; bloqueio, CAPTCHA e soft-404 viram **inconclusivo**, nunca ausência.

In [ ]:
PLATFORMS = "telegram, youtube, tiktok, linktree, x, mastodon, bluesky, devto, gitlab"  #@param {type:"string"}

from sosia.platforms import load_platforms
from sosia.net import UrlLibFetcher
from sosia.calibrate import calibrate, validate

plats = load_platforms(only=_split(PLATFORMS))
fetcher = UrlLibFetcher(min_interval=0.7, timeout=10)
usable = {}
for pid, p in plats.items():
    cal = calibrate(p, fetcher)
    val = validate(p, cal, fetcher) if cal.usable else None
    ok = cal.usable and (val is None or val.passed or cal.state.value != "differential")
    print(("OK " if ok else "-- "), f"{pid:<10} {cal.state.value:<20} {cal.reason}",
          f"| validação {val.known_hit}/{val.known_total} · {val.random_hit}/{val.random_total}" if val else "")
    if cal.usable: usable[pid] = p
print("\nutilizáveis:", list(usable))

## 6. 🌐 Varredura
Um pacote de evidências (`case.json`, `report.md`, CSV, STIX 2.1, `MANIFEST.sha256`) é gravado em `/content/runs`.

In [ ]:
BUDGET = 80  #@param {type:"integer"}
WATCH  = ""  #@param {type:"string"}

import time, pathlib
from sosia.scanner import Scanner, ScanConfig
from sosia.report import write_case, summary_counts, to_markdown, finding_line
from sosia.model import Existence

cfg = ScanConfig(budget=BUDGET, seed=1, extra_candidates=_split(WATCH), save_raw=False)
res = Scanner(brand, plats, fetcher, cfg, progress=print).run()
out = pathlib.Path("/content/runs") / f"{brand.handle}-{time.strftime('%Y%m%d-%H%M%S')}"
write_case(res, out)
t = res.stats["totals"]
print(f"\nrequisições={t['requests']} encontrados={t['found']} inconclusivos={t['inconclusive']} descartadas={t['platforms_discarded']}")
print("prioridades:", summary_counts(res))
for f in res.sorted_findings():
    if f.existence in (Existence.CONFIRMED, Existence.PROBABLE): print(finding_line(f))
print("\npacote:", out)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(to_markdown(res)))

## 7. Baixar o pacote de evidências

In [ ]:
import shutil
zip_path = shutil.make_archive(str(out), "zip", out)
try:
    from google.colab import files; files.download(zip_path)
except ImportError:
    print("fora do Colab:", zip_path)

## 8. Bench sintético (offline, ≈10 s)
Harness de regressão com verdade-terreno **simulada**: **não** são taxas de detecção do mundo real (ver `docs/BENCH_RESULTS.md`).

In [ ]:
from sosia.bench.runner import run_bench, render_markdown
from IPython.display import Markdown, display
bench = run_bench(seed=7, quick=True)
display(Markdown(render_markdown(bench)))

## 9. Medir a triagem em perfis **reais** rotulados (a prova de verdade)
Suba um JSONL (`username, display_name, bio, label` com `label` ∈ `scam|nonscam|fan|benign|reseller`, mais `brand/handle/domains`), rotulado por dois revisores.
Veja `examples/labeled_sample.jsonl`. Em linha de comando: `sosia eval rotulados.jsonl`.

In [ ]:
import json, urllib.request
SAMPLE = f"https://raw.githubusercontent.com/Ridd1kulusC0d3r/SOSIA/{BRANCH}/examples/labeled_sample.jsonl"
urllib.request.urlretrieve(SAMPLE, "/content/labeled_sample.jsonl")
!sosia eval /content/labeled_sample.jsonl